In [ ]:
#Librerias
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap



from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score




In [ ]:
# Cargamos el dataset
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/ML/churn_data.csv")  # Modificar según ruta donde se almacene

print(df.head()) # Los datos cargaron correctamente


In [ ]:
# Realizamos un breve ánalisis exploratorio
df.info()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']) # Elimnamos los nulos
df = df.drop(columns=['customerID']) # Eliminamos el ID

In [ ]:
df.describe()

No hay valores nulos en el dataset. Observamos que algunas columnas tienen un tipo de dato incorrecto, como TotalCharges, que aparece como object pero debería ser numérica. Varias variables son categóricas binarias y se convertirán a 0/1 para facilitar el análisis y el modelado.

Finalmente, las variables MonthlyCharges y TotalCharges presentan una desviación significativa respecto a su media, lo que indica una amplia dispersión en sus valores

In [ ]:
# Convertimos las variables categóricas binarias a 0 y 1,y aplicamos one-hot encoding a las variables categóricas nominales
df["Churn"] = df["Churn"].map({'Yes': 1, 'No': 0})
df["PhoneService"] = df["PhoneService"].map({'Yes': 1, 'No': 0})
df["PaperlessBilling"] = df["PaperlessBilling"].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, columns=['Contract', 'PaymentMethod'], drop_first=True)

In [ ]:
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)

In [ ]:
df["Churn"].value_counts() # El dataset esta desbalanceado

Se muestran diversas correlaciones de intensidad moderada o fuerte. Como más hay menos abandono.


In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Dividimos el data set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33, stratify=y)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20, 50],
    'min_samples_split': [2, 5, 7],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'],
    'class_weight': ['balanced']}
rf = RandomForestClassifier()
grid_rf = GridSearchCV(rf, param_grid, cv=5)
grid_rf.fit(X_train, y_train)

In [ ]:
print("Mejores parámetros:", grid_rf.best_params_)
print("Mejor precisión (CV):", grid_rf.best_score_)

In [ ]:
# Evaluamos en el test
best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)


print("Accuracy test:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

El modelo solo es capaz de detectar el 58% de los clientes que se van. Pero acierta con el 84% de los que se quedan.

In [ ]:
explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap_values_class0 = shap_values[0]
shap_values_class1 = shap_values[1]

Nos enfocaremos en la clase 1, ya que nos interesa entender por qué los clientes se van.

In [ ]:
shap_values_class1 = shap_values[:, :, 1]  # toma solo la clase 1

shap.summary_plot(shap_values_class1, X_test)

Los factores que empujan al churn son llevar poco tiempo con la compañía y facturas altas.

In [ ]:
shap.dependence_plot("tenure", shap_values_class1, X_test)

Como era de esperar, llevar poco tiempo y tener cargos elevados causa abandono, aunque también con cargos más pequeños se puede observar cierta linealidad.

In [ ]:
shap.initjs()
cliente = X_test.iloc[22]
shap.force_plot(
    explainer.expected_value[1],
    shap_values_class1[22],
    cliente
)

En este ejemplo, el modelo predice que el cliente se va.
Los factores que aumentan la probabilidad de churn son: no tener contrato anual (Contract_One year = 0, Contract_Two year = 0) y el precio mensual elevado.
Por otro lado, los factores que disminuyen la probabilidad de churn son el tiempo que lleva como cliente y el total de pagos realizados


En conclusión, vemos que las facturas bajas y llevar cierto tiempo con la compañía ayudan a que la gente se quede, y al contrario, si son altas o llevas poco tiempo, hay más riesgo de abandono.

Con SHAP puedes ver cómo influye cada variable; por ejemplo, si la variable sexo estuviera muy desbalanceada, podrías pensar que el modelo está sesgado.

Una de las grandes ventajas de SHAP es que sirve para cualquier modelo y además es muy completo, puedes mirar la influencia de las variables en todo el conjunto de datos o en casos individuales, comparar variables y queda todo muy claro y fácil de entender.